# Course 2, Week 1 — Hyperparameter Optimization

- [DeepLearning.AI platform](https://learn.deeplearning.ai/specializations/pytorch-for-deep-learning-professional-certificate/lesson)
- [Week notes](README.md)
- [GitHub issue #5](https://github.com/majorgilles/pytorch_for_deep_learning/issues/5)

**Focus:** Compare optimizers and schedulers, tune experiments, and improve model performance.

## Video guide

| Video | Covered notebook section |
|---|---|
| Choosing an evaluation objective | Accuracy, precision, recall, and F1 |

> Open the [DeepLearning.AI course lesson list](https://learn.deeplearning.ai/specializations/pytorch-for-deep-learning-professional-certificate/lesson) to watch the course video.


## Start with a clear evaluation objective

Hyperparameter optimization needs an objective: a measurable quantity that determines whether one experiment is better than another. Choose that metric before tuning, based on the real cost of each error.

For a flower classifier, overall accuracy may be a useful starting point. For a plant-disease detector, however, the consequences differ:

- A **false positive** may trigger unnecessary treatment, so precision matters.
- A **false negative** may allow disease to spread, so recall matters.

No optimizer can compensate for optimizing the wrong objective. Record both the primary metric to optimize and secondary metrics that reveal unacceptable trade-offs.


### Classification outcomes

For one positive class such as “rose,” every prediction belongs to one of four groups:

| Actual class | Predicted rose | Outcome |
|---|---|---|
| Rose | Yes | True positive ($TP$) |
| Not rose | Yes | False positive ($FP$) |
| Not rose | No | True negative ($TN$) |
| Rose | No | False negative ($FN$) |

These counts define the common metrics:

$$
\operatorname{Accuracy}=\frac{TP+TN}{TP+TN+FP+FN},
$$

$$
\operatorname{Precision}=\frac{TP}{TP+FP},
\qquad
\operatorname{Recall}=\frac{TP}{TP+FN},
$$

$$
F_1=2\frac{\operatorname{Precision}\cdot\operatorname{Recall}}
{\operatorname{Precision}+\operatorname{Recall}}.
$$

Accuracy measures all correct predictions, precision asks whether positive predictions are trustworthy, and recall asks whether actual positives were found. The harmonic mean makes $F_1$ small when either precision or recall is small.


In [1]:
import torch
import torchmetrics
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# Binary labels and predictions both have shape [N] = [8].
binary_targets: torch.Tensor = torch.tensor([1, 1, 1, 0, 0, 0, 0, 0])
binary_predictions: torch.Tensor = torch.tensor([1, 0, 1, 1, 0, 0, 0, 0])

tp: int = int(((binary_predictions == 1) & (binary_targets == 1)).sum())
fp: int = int(((binary_predictions == 1) & (binary_targets == 0)).sum())
tn: int = int(((binary_predictions == 0) & (binary_targets == 0)).sum())
fn: int = int(((binary_predictions == 0) & (binary_targets == 1)).sum())

accuracy: float = (tp + tn) / len(binary_targets)
precision: float = tp / (tp + fp)
recall: float = tp / (tp + fn)
f1: float = 2 * precision * recall / (precision + recall)

print(f"TP={tp}, FP={fp}, TN={tn}, FN={fn}")
print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1:        {f1:.3f}")

assert (tp, fp, tn, fn) == (2, 1, 4, 1)


TP=2, FP=1, TN=4, FN=1
Accuracy:  0.750
Precision: 0.667
Recall:    0.667
F1:        0.667


### Why accuracy can mislead

Suppose 95% of images show healthy plants. A model that always predicts “healthy” reaches 95% accuracy while detecting no diseased plants. Its recall for disease is zero.

This is why the metric must match the objective:

- Favor **precision** when false alarms are especially costly.
- Favor **recall** when missing a positive case is especially costly.
- Use **F1** when both error types matter and the classes are imbalanced.
- Keep **accuracy** when classes and error costs are reasonably balanced.

The decision threshold also affects precision and recall: requiring stronger evidence for a positive prediction usually raises precision while lowering recall.


### Average multiclass metrics

Multiclass metrics compute one-versus-rest outcomes for each class and then combine them:

- **Macro averaging** computes the metric per class and gives every class equal weight. Rare classes matter as much as common classes.
- **Micro averaging** pools decisions across all samples before computing the metric. Common classes therefore contribute more.

Macro metrics are useful for revealing poor performance on underrepresented species. Micro metrics summarize aggregate prediction performance. Report both when class frequencies are uneven rather than relying on one number.


In [2]:
def evaluate_metrics(
    model: nn.Module,
    val_dataloader: DataLoader,
    device: torch.device,
    num_classes: int = 10,
) -> dict[str, float]:
    """Evaluate a multiclass model over a validation dataset.

    Args:
        model: Model mapping images ``[N, ...]`` to logits ``[N, C]``.
        val_dataloader: Batches of input tensors and targets ``[N]``.
        device: Device used by both model and batches.
        num_classes: Number of output classes, ``C``.

    Returns:
        Macro-averaged accuracy, precision, recall, and F1 as Python floats.
    """
    model.eval()

    # Use the task-based TorchMetrics interfaces shown in the course.
    accuracy_metric = torchmetrics.Accuracy(
        task="multiclass", num_classes=num_classes, average="macro"
    ).to(device)
    precision_metric = torchmetrics.Precision(
        task="multiclass", num_classes=num_classes, average="macro"
    ).to(device)
    recall_metric = torchmetrics.Recall(
        task="multiclass", num_classes=num_classes, average="macro"
    ).to(device)
    f1_metric = torchmetrics.F1Score(
        task="multiclass", num_classes=num_classes, average="macro"
    ).to(device)

    with torch.no_grad():
        for images, targets in val_dataloader:
            images = images.to(device)
            targets = targets.to(device)
            logits: torch.Tensor = model(images)  # Shape [N, C].

            # TorchMetrics accepts logits and selects the largest class score.
            accuracy_metric.update(logits, targets)
            precision_metric.update(logits, targets)
            recall_metric.update(logits, targets)
            f1_metric.update(logits, targets)

    return {
        "accuracy_macro": accuracy_metric.compute().item(),
        "precision_macro": precision_metric.compute().item(),
        "recall_macro": recall_metric.compute().item(),
        "f1_macro": f1_metric.compute().item(),
    }


# Build deterministic logits [N, C] for a runnable DataLoader example.
multiclass_targets: torch.Tensor = torch.tensor([0, 0, 0, 0, 0, 1, 1, 1, 2, 2])
multiclass_predictions: torch.Tensor = torch.tensor([0, 0, 0, 0, 0, 0, 0, 1, 0, 2])
multiclass_logits: torch.Tensor = torch.full((10, 3), fill_value=-1.0)
multiclass_logits[torch.arange(10), multiclass_predictions] = 1.0

validation_dataset: TensorDataset = TensorDataset(multiclass_logits, multiclass_targets)
validation_loader: DataLoader = DataLoader(validation_dataset, batch_size=4)
identity_model: nn.Identity = nn.Identity()
evaluation_device: torch.device = torch.device("cpu")

metric_results: dict[str, float] = evaluate_metrics(
    model=identity_model,
    val_dataloader=validation_loader,
    device=evaluation_device,
    num_classes=3,
)

for metric_name, metric_value in metric_results.items():
    print(f"{metric_name:16}: {metric_value:.3f}")

assert all(isinstance(value, float) for value in metric_results.values())
assert metric_results["precision_macro"] > metric_results["recall_macro"]


accuracy_macro  : 0.611
precision_macro : 0.875
recall_macro    : 0.611
f1_macro        : 0.645


### Learning checkpoint

Before tuning a model, answer:

1. What real outcome should improve?
2. Which error is more costly: a false positive or a false negative?
3. Are classes imbalanced?
4. Should every class count equally (`macro`) or every sample count equally (`micro`)?

Then select one primary optimization metric and monitor the others as guardrails. The next step is to tune training choices against that explicit objective.
